In [15]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wk7y9wgj/unsloth_638618a1de6744c8b893cd924b482710
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wk7y9wgj/unsloth_638618a1de6744c8b893cd924b482710
  Resolved https://github.com/unslothai/unsloth.git to commit 45f060899e56d8a8d218529733075814148f6b19
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 21.9 MB/s eta 0:00:00
   

In [16]:
from google.colab import files
uploaded = files.upload()  # click and select qa_pairs.jsonl from your PC

Saving qa_pairs.jsonl to qa_pairs.jsonl


In [17]:
import os, json, torch
from pathlib import Path
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

QA_FILE    = Path("qa_pairs.jsonl")
OUTPUT_DIR = Path("vanguard-gemma3-4b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 2048
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
EPOCHS         = 3
LR             = 2e-4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/gemma-3-4b-it",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

SYSTEM_MSG = (
    "You are Vanguard, an AI assistant for an airsoft team. "
    "You have expert knowledge of airsoft tactics, gear, radio comms, milsim operations, "
    "and team coordination. Give direct, practical answers using authentic airsoft terminology."
)

def format_example(row):
    user_msg = f"{row['instruction']}\n{row.get('input','')}".strip() if row.get('input') else row['instruction']
    return {"text": (
        f"<start_of_turn>system\n{SYSTEM_MSG}<end_of_turn>\n"
        f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{row['output']}<end_of_turn>"
    )}

raw_pairs = [json.loads(l) for l in QA_FILE.read_text().splitlines() if l.strip()]
print(f"Loaded {len(raw_pairs)} Q&A pairs")

dataset  = Dataset.from_list([format_example(p) for p in raw_pairs]).train_test_split(test_size=0.05, seed=42)
train_ds = dataset["train"]
eval_ds  = dataset["test"]

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=eval_ds,
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR/"checkpoints"),
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=True,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, learning_rate=LR,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, eval_strategy="steps", eval_steps=50,
        save_strategy="steps", save_steps=100, save_total_limit=2,
        warmup_ratio=0.05, lr_scheduler_type="cosine", optim="adamw_8bit",
        report_to="none", seed=42,
    ),
)
trainer.train()

merged_dir = OUTPUT_DIR / "merged"
merged_dir.mkdir(exist_ok=True)
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
print("Done — model saved to vanguard-gemma3-4b/merged/")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.8: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [07:18<07:18, 438.94s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [15:34<00:00, 467.06s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:48<00:00, 114.09s/it]


Unsloth: Merge process complete. Saved to `/content/vanguard-gemma3-4b/merged`
Done — model saved to vanguard-gemma3-4b/merged/


In [3]:
from google.colab import files
!zip -r vanguard-merged.zip vanguard-gemma3-4b/merged/
files.download("vanguard-merged.zip")

updating: vanguard-gemma3-4b/merged/ (stored 0%)
updating: vanguard-gemma3-4b/merged/.cache/ (stored 0%)
updating: vanguard-gemma3-4b/merged/.cache/huggingface/ (stored 0%)
updating: vanguard-gemma3-4b/merged/.cache/huggingface/.gitignore (stored 0%)
updating: vanguard-gemma3-4b/merged/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
updating: vanguard-gemma3-4b/merged/.cache/huggingface/download/ (stored 0%)
updating: vanguard-gemma3-4b/merged/.cache/huggingface/download/model-00002-of-00002.safetensors.metadata (deflated 30%)
updating: vanguard-gemma3-4b/merged/.cache/huggingface/download/model.safetensors.index.json.metadata (deflated 27%)
updating: vanguard-gemma3-4b/merged/.cache/huggingface/download/model-00001-of-00002.safetensors.metadata (deflated 30%)
updating: vanguard-gemma3-4b/merged/generation_config.json (deflated 34%)
updating: vanguard-gemma3-4b/merged/model.safetensors.index.json (deflated 97%)
updating: vanguard-gemma3-4b/merged/model-00001-of-00002.safetensors (deflat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>